### Rete neurale generativa per simulazioni (addestrata su Geant4) - parte di training

Spostiamoci su gpu (se esiste) e carichiamo i dati
- carichiamo un pezzo di file alla volta (chunk) per non appesantire troppo la memoria
- filtriamo già per caricare solo i primari (da cambiare in seguito, per un primo modello semplificato va bene)

In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Using device: cpu


In [2]:
import pandas as pd

def load_primary_tracks(file_path, chunksize=100_000):
    primary_rows = []

    # Lettura a chunk per gestire file grandi
    for chunk in pd.read_csv(file_path, sep="\t", chunksize=chunksize):
        # Filtro dei primari (ParentID == 0)
        primaries = chunk[chunk["ParentID"] == 0]
        primary_rows.append(primaries)

    # Concatenazione finale
    df_primaries = pd.concat(primary_rows, ignore_index=True)
    return df_primaries

# Esempio d'uso
file_path = "DatiPerNN_BigMixture_1500keV_PulseTube_1e4EInt.txt"  
df = load_primary_tracks(file_path)

# Anteprima
print(df.head())

   EventNumber  ParentID  TrackID ParticleName  ParticleEn(MeV)  Edep(MeV)  \
0            0         0        1           e-              1.5   0.581270   
1            0         0        1           e-              1.5   0.396748   
2            0         0        1           e-              1.5   0.091727   
3            0         0        1           e-              1.5   0.052685   
4            0         0        1           e-              1.5   0.058514   

      X(mm)     Y(mm)     Z(mm)  Step(mm)  Track(mm)     VolumeName  \
0 -0.004809  0.182232  0.060429  0.198935   0.198935  crystal_phys0   
1 -0.027830  0.335070  0.121910  0.292686   0.491621  crystal_phys0   
2 -0.007447  0.399106  0.152664  0.142127   0.633748  crystal_phys0   
3 -0.046494  0.394904  0.190319  0.108555   0.742303  crystal_phys0   
4 -0.012974  0.371026  0.204682  0.089589   0.831892  crystal_phys0   

  MotherVolumeName  CenterX(mm)  CenterY(mm)  CenterZ(mm)  
0           Module            0            0

Trasformiamo i dati stringa in dati numero da poter usare nell'addestramento
- usiamo LabelEncoder per VolumeName e MotherVolumeName, che trasforma le categorie testuali in categorie di interi
- verifichiamo i dtype

In [3]:
from sklearn.preprocessing import LabelEncoder

le_part = LabelEncoder()
le_vol = LabelEncoder()
le_moth = LabelEncoder()

df_encoded = df.copy()

df_encoded["ParticleName"] = le_part.fit_transform(df_encoded["ParticleName"])
df_encoded["VolumeName"] = le_vol.fit_transform(df_encoded["VolumeName"])
df_encoded["MotherVolumeName"] = le_moth.fit_transform(df_encoded["MotherVolumeName"])

df_encoded.rename(columns={
    "ParticleName": "ParticleName_enc",
    "VolumeName": "VolumeName_enc",
    "MotherVolumeName": "MotherVolumeName_enc",

}, inplace=True)

print(df_encoded.dtypes)

EventNumber               int64
ParentID                  int64
TrackID                   int64
ParticleName_enc          int64
ParticleEn(MeV)         float64
Edep(MeV)               float64
X(mm)                   float64
Y(mm)                   float64
Z(mm)                   float64
Step(mm)                float64
Track(mm)               float64
VolumeName_enc            int64
MotherVolumeName_enc      int64
CenterX(mm)               int64
CenterY(mm)               int64
CenterZ(mm)               int64
dtype: object


Normalizziamo i dati tra [0, 1] 

- Rinominiamo le colonne
- Salviamo il fattore di normalizzazione (scaler). Questo è fondamentale per riconvertire i dati generati dalla rete (che saranno nello spazio della normalizzazione) in dati che abbiano senso fisico

In [4]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Colonne da normalizzare
cols_to_normalize = [
    "ParticleEn(MeV)", 
    "Edep(MeV)", 
    "X(mm)", 
    "Y(mm)", 
    "Z(mm)", 
    "Step(mm)", 
    "Track(mm)"
]

scalers = {}  # oppure una lista se preferisci, ma un dizionario è più leggibile

# Applica uno scaler per ogni colonna
for col in cols_to_normalize:
    values = df_encoded[[col]].values
    if np.all(values == values[0]):  # tutti i valori uguali
        # Creo uno scaler finto compatibile con MinMaxScaler
        scaler = MinMaxScaler()
        scaler.min_ = np.array([0.0])
        scaler.scale_ = np.array([1.0])  # dividendo per sé stesso non cambia i valori normalizzati
        scaler.data_min_ = np.array([values[0][0]])
        scaler.data_max_ = np.array([values[0][0]])
        scaler.data_range_ = np.array([1.0])  # per evitare divisione per zero interna
        df_encoded[[col]] = values / values[0][0]  # normalizzazione semplice
    else:
        scaler = MinMaxScaler()
        df_encoded[[col]] = scaler.fit_transform(values)

    scalers[col] = scaler

# Rinomina le colonne
df_encoded.rename(columns={
    "ParticleEn(MeV)": "ParticleEn",
    "Edep(MeV)": "Edep",
    "X(mm)": "X",
    "Y(mm)": "Y",
    "Z(mm)": "Z",
    "Step(mm)": "Step",
    "Track(mm)": "Track"
}, inplace=True)


Raggruppiamo per EventNumber
- creiamo una lista in cui ogni entrata corrisponde a un evento con tutti i suoi step
- per come Geant4 lavora, ogni evento dovrebbe essere già ordinato per traccia

In [5]:
grouped = df_encoded.groupby("EventNumber", sort=False)
event_numbers = list(grouped.groups.keys())

In [6]:
chosen_event = 0 

if chosen_event in event_numbers:
    event_df = grouped.get_group(chosen_event)
    print(f"Event {chosen_event}:\n", event_df.to_string(index=False))
else:
    print(f"L'evento {chosen_event} non esiste nel dataset.")

Event 0:
  EventNumber  ParentID  TrackID  ParticleName_enc  ParticleEn     Edep        X        Y        Z     Step    Track  VolumeName_enc  MotherVolumeName_enc  CenterX(mm)  CenterY(mm)  CenterZ(mm)
           0         0        1                 0         1.0 0.448602 0.489443 0.590534 0.534947 0.391159 0.062356               0                     0            0            0            0
           0         0        1                 0         1.0 0.306192 0.479200 0.654117 0.562157 0.575502 0.154123               0                     0            0            0            0
           0         0        1                 0         1.0 0.070783 0.488269 0.680757 0.575768 0.279458 0.198684               0                     0            0            0            0
           0         0        1                 0         1.0 0.040650 0.470896 0.679009 0.592432 0.213445 0.232720               0                     0            0            0            0
           0         0   

Riorganizziamo i nostri dati in strutture numpy per prepararli all'uso con PyTorch.

- Per ogni evento costruiamo una tupla con l’energia iniziale della particella primaria (ParticleEn(MeV)), e una sequenza di vettori contenenti le caratteristiche step-by-step (Edep, X, Y, Z, Step, VolumeName, MotherVolumeName).
- Ogni evento diventa quindi una tupla: (energia_iniziale, sequenza) dove sequenza è un array NxD (N = numero di step, D = numero di feature).
- In questo modello semplificato consideriamo solo un sottoinsieme delle colonne disponibili. Eventualmente sarà da estendere.

In [7]:
import numpy as np

# Colonne di interesse
cols = ["EventNumber", "ParticleEn", "Edep", "X", "Y", "Z", "Step", "VolumeName_enc", "MotherVolumeName_enc"]

# Estrai sotto-DataFrame ridotto e converti a numpy
arr = df_encoded[cols].to_numpy()

# Ricava la lista di eventi (con energy iniziale e sequenza) direttamente da numpy
all_events = []
current_event = None
current_seq = []

for row in arr:
    event_num = row[0]
    energy = row[1]
    features = row[2:]  # le altre feature
    
    if event_num != current_event:
        # salvo il precedente evento se esiste
        if current_event is not None:
            all_events.append((energy_init, np.array(current_seq, dtype=np.float32)))
        current_event = event_num
        energy_init = energy
        current_seq = []
    current_seq.append(features)

# Aggiungi l’ultimo evento
if current_event is not None:
    all_events.append((energy_init, np.array(current_seq, dtype=np.float32)))


In [8]:
# Prendo il primo evento dalla lista
evento_0 = all_events[0]

# Estraggo l'energia iniziale
energia_iniziale = evento_0[0]

# Estraggo la matrice delle features (sequenza)
sequenza_features = evento_0[1]

# Stampo i risultati in modo chiaro
print(f"Energia iniziale del primo evento: {energia_iniziale} MeV")
print(f"Dimensioni della sequenza (step, feature): {sequenza_features.shape}")
print("Prime righe della sequenza (features):")
print(sequenza_features[:5])

Energia iniziale del primo evento: 1.0 MeV
Dimensioni della sequenza (step, feature): (14, 7)
Prime righe della sequenza (features):
[[0.448602   0.4894427  0.590534   0.53494716 0.39115945 0.
  0.        ]
 [0.30619165 0.47919992 0.65411687 0.56215674 0.57550186 0.
  0.        ]
 [0.07078264 0.48826903 0.6807568  0.5757675  0.279458   0.
  0.        ]
 [0.04065043 0.47089612 0.6790087  0.5924325  0.21344543 0.
  0.        ]
 [0.04514913 0.4858097  0.66907513 0.59878916 0.17615163 0.
  0.        ]]


Dividiamo i vettori in train, test e validation

- dividiamo in 90% train e 10% test

In [9]:
from sklearn.model_selection import train_test_split

train_events, test_events = train_test_split(all_events, test_size=0.2, random_state=42)

I vettori potrebbero non avere tutti la stessa lunghezza. Facciamo padding

- convertiamo le tuple (energy_init, sequenza) create in tensori pytorch
- aggiungiamo degli zeri ai vettori più corti
- creiamo una maschera per dire alla rete di ignorare i valori aggiunti con il padding

In [10]:
import torch
from torch.nn.utils.rnn import pad_sequence

# Sequenze e energie
train_sequences = [torch.tensor(event[1], dtype=torch.float32) for event in train_events]
train_energies = [event[0] for event in train_events]
test_sequences = [torch.tensor(event[1], dtype=torch.float32) for event in test_events]
test_energies = [event[0] for event in test_events]

# Padding
train_padded = pad_sequence(train_sequences, batch_first=True, padding_value=-1.0)
test_padded = pad_sequence(test_sequences, batch_first=True, padding_value=-1.0)

# Maschera
train_mask = (train_padded[:, :, 0] != -1)
test_mask = (test_padded[:, :, 0] != -1)

# Trasferiamo su GPU se disponibile
train_padded = train_padded.to(device)
test_padded = test_padded.to(device)
train_mask = train_mask.to(device)
test_mask = test_mask.to(device)

Creiamo il dataset di tensori e passiamolo a pytorch con un dataloader

- ricordiamoci che sequences l'abbiamo già trasformato in un tensore nella cella precedente

In [11]:
from torch.utils.data import TensorDataset, DataLoader

# Tensore energie
train_energy_tensor = torch.tensor(train_energies, dtype=torch.float32).to(device)
test_energy_tensor = torch.tensor(test_energies, dtype=torch.float32).to(device)

# Dataset e DataLoader 
train_dataset = TensorDataset(train_padded, train_energy_tensor, train_mask)
train_dataloader = DataLoader(train_dataset, batch_size=1000, shuffle=True)
test_dataset = TensorDataset(test_padded, test_energy_tensor, test_mask)
test_dataloader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# --- Verifica un batch train ---
for batch_seq, batch_energy, batch_mask in train_dataloader:
    print("Train batch shapes:", batch_seq.shape, batch_energy.shape, batch_mask.shape)
    print("Train batch device:", batch_seq.device, batch_energy.device, batch_mask.device)
    break

# --- Verifica un batch test ---
for batch_seq, batch_energy, batch_mask in test_dataloader:
    print("Test batch shapes:", batch_seq.shape, batch_energy.shape, batch_mask.shape)
    print("Test batch device:", batch_seq.device, batch_energy.device, batch_mask.device)
    break

Train batch shapes: torch.Size([1000, 22, 7]) torch.Size([1000]) torch.Size([1000, 22])
Train batch device: cpu cpu cpu
Test batch shapes: torch.Size([1000, 22, 7]) torch.Size([1000]) torch.Size([1000, 22])
Test batch device: cpu cpu cpu


Importiamo il modello

- modello ibrido GRU (Gated Recurrent Unit) + Transformer con chunking adattivo
- il chunking serve per dividere le sequenze e passare alla rete solo un certo numero di step per volta. Sostanzialmente è simile al concetto del minibatch, ma mentre il minibatch suddivide gli eventi, il chunk suddivide gli step dell'evento
- La GRU è una rete ricorrente che usa due porte — update e reset — per decidere quanto mantenere o dimenticare dello stato precedente, aggiornando così lo stato nascosto in modo efficiente e prevenendo il problema del vanishing gradient su sequenze lunghe.
- Il Transformer usa meccanismi di self-attention per pesare dinamicamente l'importanza di ogni elemento della sequenza rispetto agli altri, permettendo di catturare dipendenze a lungo raggio in parallelo, senza ricorrere a passaggi sequenziali.

In [12]:
from model_architecture import GRUTransformerGenerator

Impostiamo i parametri del modello

- hidden_dim è il numero di neuroni nascosti con cui la rete rappresenta le nostre features reali
- gru_layers è il numero di gru che vogliamo; 1 è sufficiente
- nhead è il numero di “teste di attenzione” nel transformer. Ogni testa lavora in parallelo per catturare relazioni diverse tra i passi della sequenza
- num_transformer_layers è il numero di transformers che vogliamo usare. Più transformers aumentano la capacità del modello

In [13]:
input_dim = train_padded.shape[-1] 
n_volume_classes = len(le_vol.classes_)
n_mother_volume_classes = len(le_moth.classes_)

model = GRUTransformerGenerator(
    input_dim=input_dim,
    hidden_dim=64,
    energy_dim=1,
    gru_layers=1,
    nhead=1,
    num_transformer_layers=1,
    dropout=0.4,
    max_chunk_len=100,
    n_volume_classes=n_volume_classes,           
    n_mother_volume_classes=n_mother_volume_classes  
).to(device)


/home/emanuelemazzola/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


In [14]:
for batch_seq, batch_energy, batch_mask in train_dataloader: 
    output, _ = model(batch_seq, batch_energy, batch_mask)
    print("Input shape: ", batch_seq.shape)
    print("Output shape:", output.shape)
    break

Input shape:  torch.Size([1000, 22, 7])
Output shape: torch.Size([1000, 22, 12])


Creiamo optimizer e loss

- usiamo la log-likelihood negativa di una gaussiana (NLL) e non MSE per catturare la varianza dei dati invece che mediare
- la loss sarà mista: NLL per dati continui (Edep, X, Y, Z, Step), CrossEntropy per dati categorici (VolumeName, MotherVolumeName)
- questo perchè la gasussian null restituisce dei numeri razionali, che non vanno bene quando bisogna decidere le classi di appartenenza (nel nostro caso i volumi), perchè poi andrebbero arrotondati arbitrariamente
- criterion è solo un wrapper per richiamare la loss più facilmente

In [15]:
import torch.nn.functional as F
import numpy as np
import torch

def gaussian_nll(mu, logvar, target):

    inv_var = torch.exp(-logvar)
    mse = (target - mu) ** 2
    nll = 0.5 * (mse * inv_var + logvar + np.log(2 * np.pi))
    return nll.sum(dim=-1).mean()  # somma sulle feature, media su batch/seq


def custom_loss(preds, targets, mask, n_vol_classes, n_mother_classes):


    # Estrazione delle predizioni
    pred_cont = preds[:, :, :10]  # 2*n_cont (mu + logvar), con n_cont=5
    pred_vol = preds[:, :, 10:10 + n_vol_classes]
    pred_mother = preds[:, :, 10 + n_vol_classes:]

    # Estrazione dei target
    target_cont = targets[:, :, :5]
    target_vol = targets[:, :, 5].long()
    target_mother = targets[:, :, 6].long()

    # Split di mu e logvar per le variabili continue
    mu = pred_cont[:, :, :5]
    logvar = pred_cont[:, :, 5:]

    # Maschera per valori validi (diversi da -1)
    valid_mask = (target_cont != -1).all(dim=-1) & mask  # [batch, seq]

    # Applica la maschera a mu, logvar e target_cont
    mu_valid = mu[valid_mask]
    logvar_valid = logvar[valid_mask]
    target_valid = target_cont[valid_mask]

    # Gaussian NLL solo sugli step validi
    if mu_valid.numel() > 0:
        loss_cont = gaussian_nll(
            mu_valid.unsqueeze(0), 
            logvar_valid.unsqueeze(0), 
            target_valid.unsqueeze(0)
        )
    else:
        loss_cont = torch.tensor(0.0, device=preds.device)

    ### CrossEntropy ###
    ce_loss_vol = F.cross_entropy(
        pred_vol.reshape(-1, n_vol_classes),
        target_vol.reshape(-1),
        reduction='mean',
        ignore_index=-1
    )

    ce_loss_mother = F.cross_entropy(
        pred_mother.reshape(-1, n_mother_classes),
        target_mother.reshape(-1),
        reduction='mean',
        ignore_index=-1
    )

    return loss_cont + ce_loss_vol + ce_loss_mother


In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = lambda preds, targets, mask: custom_loss(preds, targets, mask, n_volume_classes, n_mother_volume_classes)

Creiamo il loop di training

- concetto fondamentale per cui tutto funziona: dato un passo iniziale, la rete predice il passo successivo, poi calcola la loss tra il passo successivo predetto e il passo successivo reale
- creiamo quindi i target (i passi successivi) con cui confrontare quelli predetti
- creiamo poi il loop di training
- nel training c'è early stopping: se la loss di validazione non migliora per 5 epoche di fila, il training termina
- salviamo sia i migliori pesi che gli encoder necessari a riconvertire gli output numerici in parole (tipo di particella, volumi ecc)

In [17]:
def create_shifted_targets(sequences):

    target = sequences[:, 1:, :]
    pad = -torch.ones(sequences.size(0), 1, sequences.size(2), dtype=sequences.dtype, device=sequences.device)
    target = torch.cat([target, pad], dim=1)
    return target

# target_sequences = create_shifted_targets(padded_sequences)


In [18]:
import pickle

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=50, patience=5, encoders_scalers=None):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(num_epochs):
        # TRAINING
        model.train()
        train_loss = 0
        for batch_seq, batch_energy, batch_mask in train_loader:
            optimizer.zero_grad()
            outputs, _ = model(batch_seq, batch_energy, batch_mask)
            
            targets = create_shifted_targets(batch_seq)
            loss = criterion(outputs, targets, batch_mask)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # VALIDAZIONE
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_seq, batch_energy, batch_mask in val_loader:
                outputs, _ = model(batch_seq, batch_energy, batch_mask)
                targets = create_shifted_targets(batch_seq)
                loss = criterion(outputs, targets, batch_mask)  
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # EARLY STOPPING
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_model_weights.pt")  # salva i pesi migliori
            #torch.save(model, "complete_model.pt") # salva il modello con i pesi migliori
            if encoders_scalers is not None:    # salva encoders e scalers, se passati
                with open("encoders_scalers.pkl", "wb") as f:
                    pickle.dump(encoders_scalers, f)
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f"Early stopping! Val loss non migliora da {epochs_no_improve} epoche consecutive.")
                break
    
    return model

In [19]:
encoders_scalers = {
    "le_part": le_part,
    "le_vol": le_vol,
    "le_moth": le_moth,
    "scalers": scalers
}

trained_model = train_model(model, train_dataloader, test_dataloader, criterion, optimizer, num_epochs=30, patience=10, encoders_scalers=encoders_scalers)

Epoch 1, Train Loss: 4.1805, Val Loss: 1.4637
Epoch 2, Train Loss: 1.3525, Val Loss: -0.3552
Epoch 3, Train Loss: -0.2720, Val Loss: -1.6996
Epoch 4, Train Loss: -1.9661, Val Loss: -3.0503
Epoch 5, Train Loss: -3.4534, Val Loss: -4.3327
Epoch 6, Train Loss: -4.6772, Val Loss: -5.4265
Epoch 7, Train Loss: -5.6403, Val Loss: -6.1702
Epoch 8, Train Loss: -6.3047, Val Loss: -6.6868
Epoch 9, Train Loss: -6.7774, Val Loss: -7.2776
Epoch 10, Train Loss: -7.1048, Val Loss: -7.3654
Epoch 11, Train Loss: -7.3296, Val Loss: -7.6567
Epoch 12, Train Loss: -7.6550, Val Loss: -8.0089
Epoch 13, Train Loss: -7.8142, Val Loss: -8.2485
Epoch 14, Train Loss: -8.0288, Val Loss: -8.4277
Epoch 15, Train Loss: -8.2685, Val Loss: -8.4141
Epoch 16, Train Loss: -8.3261, Val Loss: -8.7501
Epoch 17, Train Loss: -8.5108, Val Loss: -8.9503
Epoch 18, Train Loss: -8.6422, Val Loss: -8.9737
Epoch 19, Train Loss: -8.7481, Val Loss: -8.8727
Epoch 20, Train Loss: -8.4182, Val Loss: -8.9303
Epoch 21, Train Loss: -8.8898, V